# 机器学习基础项目 — 回归（Regressão）中文注释版

**小组 22**

### 成员
- **jiyi Li** (62244)
- **oujie Wu** (62228)
- **josé lourenço** (62817)

> 本文件是 `projeto_G22_Regressao.ipynb` 的中文版，用于复习。代码完全相同，注释和说明已翻译为中文。

## 问题定义

**根据学生行为变量预测期末考试成绩（exam_score）**

排除以下变量（会造成数据泄露或与目标直接相关）：
- `student_id`（学生ID，无信息量）
- `productivity_score`（直接派生自目标）
- `burnout_level`（直接派生自目标）
- `focus_index`（直接派生自目标）

## 目标变量（Target）

### 回归场景（y）

预测 **exam_score**（考试分数，连续值 0~100）

## 利益相关者（Stakeholders）

- **学生本人**：了解生活习惯（睡眠、屏幕时间、学习/娱乐平衡）如何直接影响学业成绩，从而主动调整行为，避免不及格或极度疲惫（burnout）

- **学校/教育机构**：提前识别有不及格风险的学生，合理分配教学资源（额外辅导班、调整作业量）

- **辅导员/导师**：基于数据提供个性化建议，而非泛泛而谈——聚焦模型识别出的具体问题变量（如游戏时间过多、睡眠质量差）

- **学生心理支持服务**：提前干预——当学生同时呈现低学业成绩 + 心理健康问题指标时（高咖啡因摄入 + 睡眠不足），主动提供心理支持

## 1. 数据预处理

读取数据、处理缺失值、划分训练集/测试集、编码类别特征

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

# 读取含缺失值的数据集
df_student_records = pd.read_csv('../student_records_missing.csv')
df_colunas = df_student_records.columns

# 打印每列缺失值比例
missing_percent = df_student_records.isnull().mean() * 100
missing_percent = missing_percent.sort_values(ascending=False)
print("--- 填充前各列缺失值比例 (%) ---")
print(missing_percent[missing_percent > 0])

# 定义目标变量 y
y = df_student_records["exam_score"].copy()

# 定义特征矩阵 X（删除不用的列）
df_student_records.drop(
    columns=["student_id", "exam_score", "productivity_score", "burnout_level", "focus_index"],
    inplace=True
)
X = df_student_records.copy()

# 划分训练集（80%）和测试集（20%），random_state=7 保证可复现
X_learning, X_test, y_learning, y_test = train_test_split(
    X, y, test_size=0.2, random_state=7
)

# 区分数值列和类别列
num_cols = X_learning.select_dtypes(include=["float64", "int64"]).columns.tolist()
cat_cols = X_learning.select_dtypes(include=["object", "string"]).columns.tolist()

# 用训练集的中位数填充数值缺失值（中位数比均值对异常值更鲁棒）
medians = X_learning[num_cols].median()
# 用训练集的众数填充类别缺失值
modes = X_learning[cat_cols].mode().iloc[0]

# 填充训练集
X_learning[num_cols] = X_learning[num_cols].fillna(medians)
X_learning[cat_cols] = X_learning[cat_cols].fillna(modes)

# 用训练集的统计量填充测试集（绝对不能用测试集自己的统计量！防止数据泄露）
X_test[num_cols] = X_test[num_cols].fillna(medians)
X_test[cat_cols] = X_test[cat_cols].fillna(modes)

print("\n填充后总缺失值数量：", X_learning.isnull().sum().sum() + X_test.isnull().sum().sum())

# One-Hot Encoding 处理类别特征（gender、academic_level、internet_quality）
# 选择 One-Hot 而非 Ordinal：这三个特征没有内在顺序，Ordinal 会引入虚假的大小关系
X_learning = pd.get_dummies(X_learning, columns=cat_cols)
X_test = pd.get_dummies(X_test, columns=cat_cols)


## 2. 特征选择与降维（Feature Selection）

基于 TP05 的五种方法，适配回归任务。

**核心原则**：所有特征选择操作只在 `X_learning` 上 fit，再 transform `X_test`，防止数据泄露。

**为什么要做特征选择？**
- 移除冗余特征 → 降低过拟合风险
- 降低维度 → 加速训练（特别是 KNN）
- 提升模型可解释性

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression, SequentialFeatureSelector
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# SFS 和 PCA 需要先标准化（StandardScaler：z = (x - μ) / σ）
scaler_pre = StandardScaler()
X_learn_scaled = scaler_pre.fit_transform(X_learning)  # fit+transform 训练集
X_test_scaled  = scaler_pre.transform(X_test)          # 只 transform 测试集

# ══ 方法A：方差阈值（Variance Threshold） ══════════════════════════════════
# 原理：方差极小的特征几乎不变化 → 信息量低 → 删除
# threshold=0.01：方差低于0.01的特征被删除
var_sel = VarianceThreshold(threshold=0.01)
X_learn_var = pd.DataFrame(
    var_sel.fit_transform(X_learning),
    columns=X_learning.columns[var_sel.get_support()]
)
X_test_var = pd.DataFrame(
    var_sel.transform(X_test),
    columns=X_learning.columns[var_sel.get_support()]
)
print(f"方法A - 方差阈值: {X_learning.shape[1]} → {X_learn_var.shape[1]} 个特征")

# ══ 方法B：相关性过滤（Correlation Filter） ═══════════════════════════════
# 原理：两个特征高度相关（>0.9）→ 信息重复 → 保留一个，删除另一个
corr_matrix = X_learning.corr().abs()
# 只看上三角矩阵（避免重复比较）
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop_corr = [c for c in upper.columns if any(upper[c] > 0.9)]
X_learn_corr = X_learning.drop(columns=to_drop_corr)
X_test_corr  = X_test.drop(columns=to_drop_corr)
print(f"方法B - 相关性过滤: {X_learning.shape[1]} → {X_learn_corr.shape[1]} 个特征")
print(f"  删除的特征: {to_drop_corr}")

# 绘制相关矩阵热力图
plt.figure(figsize=(10, 8))
plt.imshow(corr_matrix, cmap="viridis", aspect="auto")
plt.colorbar(label="绝对相关系数")
plt.title("特征相关矩阵（绝对值）")
plt.xticks(range(len(X_learning.columns)), X_learning.columns, rotation=90, fontsize=7)
plt.yticks(range(len(X_learning.columns)), X_learning.columns, fontsize=7)
plt.tight_layout()
plt.show()

# ══ 方法C：互信息（Mutual Information） ══════════════════════════════════
# 原理：衡量特征与目标y之间的统计依赖，能捕捉非线性关系
# 回归用 mutual_info_regression（分类用 mutual_info_classif）
mi_scores = mutual_info_regression(X_learning, y_learning, random_state=7)
mi_series = pd.Series(mi_scores, index=X_learning.columns).sort_values(ascending=False)
top_mi = mi_series.head(10).index  # 保留得分最高的10个特征
X_learn_mi = X_learning[top_mi]
X_test_mi  = X_test[top_mi]
print(f"\n方法C - 互信息（top 10）: {top_mi.tolist()}")

# 绘制互信息得分柱状图
mi_series.sort_values().plot(kind="barh", figsize=(8, 6))
plt.title("互信息得分（Mutual Information Score）— 回归")
plt.xlabel("得分")
plt.tight_layout()
plt.show()

# ══ 方法D：顺序特征选择（Sequential Feature Selection） ══════════════════
# 原理：贪心算法，每次加入使模型RMSE最小的特征，共选8个
# 属于 Wrapper 方法：把模型包在里面评估特征
# 用 LinearRegression 作为评估器（中性、快速）
sfs = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select=8,      # 选8个特征
    direction="forward",          # 正向选择（从0个开始加）
    scoring="neg_root_mean_squared_error",  # 用RMSE评估
    cv=3,                         # 3折交叉验证
    n_jobs=-1                     # 并行计算
)
sfs.fit(X_learn_scaled, y_learning)
sfs_cols = X_learning.columns[sfs.get_support()]
X_learn_sfs = X_learning[sfs_cols]
X_test_sfs  = X_test[sfs_cols]
print(f"\n方法D - SFS（8个特征）: {sfs_cols.tolist()}")

# ══ 方法E：主成分分析（PCA） ══════════════════════════════════════════════
# 原理：不选原始特征，而是创造新的主成分（原特征的线性组合）
# 选取能解释≥95%方差的最少主成分数
# 注意：PCA必须在标准化数据上进行！
pca_full = PCA().fit(X_learn_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_components = int(np.argmax(cumvar >= 0.95)) + 1
print(f"\n方法E - PCA: {n_components} 个主成分解释 ≥95% 的方差")

pca = PCA(n_components=n_components)
X_learn_pca = pca.fit_transform(X_learn_scaled)
X_test_pca  = pca.transform(X_test_scaled)

# 绘制PCA累积方差曲线
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(cumvar)+1), cumvar, marker="o")
plt.axhline(0.95, linestyle="--", label="95% 方差")
plt.xlabel("主成分数量")
plt.ylabel("累积解释方差")
plt.title("PCA — 累积解释方差")
plt.legend()
plt.tight_layout()
plt.show()


### 特征集对比辅助函数

每个模型都调用此函数，在6个特征集上用相同的默认参数评估，
以隔离特征选择对性能的影响（排除超参数调优的干扰）。

In [ ]:
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 全局字典：6个特征集（训练集，测试集）
FEATURE_SETS = {
    "Baseline（原始全特征）":  (X_learning,   X_test),
    "方法A: 方差阈值":         (X_learn_var,  X_test_var),
    "方法B: 相关性过滤":       (X_learn_corr, X_test_corr),
    "方法C: 互信息":           (X_learn_mi,   X_test_mi),
    "方法D: SFS":              (X_learn_sfs,  X_test_sfs),
    "方法E: PCA":              (X_learn_pca,  X_test_pca),
}

def comparar_feature_sets(model, nome_modelo):
    """在每个特征集上训练模型，返回对比表（按RMSE升序排列）"""
    resultados = []
    for nome, (Xl, Xt) in FEATURE_SETS.items():
        # 每次重新fit StandardScaler（只用该特征集的训练数据）
        sc = StandardScaler()
        Xl_s = sc.fit_transform(Xl)
        Xt_s = sc.transform(Xt)

        # clone(model)：每次使用全新的模型实例，避免状态污染
        m = clone(model)
        m.fit(Xl_s, y_learning)
        preds = m.predict(Xt_s)

        resultados.append({
            "特征集":     nome,
            "特征数量":   Xl_s.shape[1],
            "RMSE（↓）": round(np.sqrt(mean_squared_error(y_test, preds)), 3),
            "MAE（↓）":  round(mean_absolute_error(y_test, preds), 3),
            "R²（↑）":   round(r2_score(y_test, preds), 3),
        })

    df = pd.DataFrame(resultados).sort_values("RMSE（↓）").reset_index(drop=True)
    print(f"\n=== {nome_modelo} — 特征选择对比表 ===")
    print(df.to_string(index=False))
    return df


## 3. 决策树回归（Decision Tree Regressor）

**原理**：每个节点选择一个特征和分割点，使两个子集的 squared_error 之和最小。
叶节点的预测值 = 该叶节点训练样本的 y 均值。

**为什么不太需要特征选择**：决策树在每个节点自动选最有用的特征，无用特征不会被选中分裂。

### 3.1 各特征集对比

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# 用默认参数在6个特征集上评估，观察特征选择的效果
dt_fs_results = comparar_feature_sets(
    DecisionTreeRegressor(random_state=7), "决策树回归"
)


### 3.2 GridSearchCV — 超参数调优（全特征集）

**参数含义**：
- `criterion`：分裂标准（squared_error = 最小化均方误差，absolute_error = 最小化平均绝对误差）
- `max_depth`：树的最大深度（越深越容易过拟合，None = 不限制）
- `min_samples_split`：分裂节点所需最少样本数
- `min_samples_leaf`：叶节点所需最少样本数

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

param_grid = {
    "criterion":         ["squared_error", "absolute_error"],
    "max_depth":         [1, 2, 3, 4, 5, 6, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf":  [1, 2, 5, 10],
}

# GridSearchCV：对param_grid的所有组合做交叉验证，选最优
grid = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=7),
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",  # sklearn用负值，所以要取反
    cv=KFold(n_splits=5, shuffle=True, random_state=7),  # 5折交叉验证
    n_jobs=-1  # 并行计算
)

grid.fit(X_learning, y_learning)  # 在训练集上训练和验证

print("最优参数:", grid.best_params_)
# best_score_ 是负值（因为scoring用的是 neg_），取反得到正RMSE
print("交叉验证最优 RMSE:", round(-grid.best_score_, 3))

# 用最优模型在测试集上评估（只评估一次！不能用测试集选参数）
final_model = grid.best_estimator_
y_test_pred = final_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae  = mean_absolute_error(y_test, y_test_pred)
r2   = r2_score(y_test, y_test_pred)

print("\n--- 测试集指标 ---")
print(f"RMSE（均方根误差，越小越好）: {round(rmse, 3)}")
print(f"MAE（平均绝对误差，越小越好）: {round(mae, 3)}")
print(f"R²（决定系数，越接近1越好）: {round(r2, 3)}")


### 3.3 决策树可视化

In [ ]:
from matplotlib import pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(120, 30))
plot_tree(
    final_model,
    feature_names=X.columns,       # 显示特征名
    class_names=df_colunas,
    filled=True,                    # 节点用颜色填充
    rounded=True,
    fontsize=8,
)
plt.title("最终决策树（GridSearchCV 最优参数）")
plt.show()


## 4. 线性回归（Linear Regression）

**原理**：`y = α + β₁x₁ + β₂x₂ + ... + βₙxₙ`
通过最小二乘法找到使残差平方和最小的系数。

**为什么非常需要特征选择和StandardScaler**：
- 高度相关的特征（多重共线性）会让系数不稳定
- 不缩放的话，大尺度特征主导系数，小尺度特征被忽视

### 4.1 各特征集对比

In [ ]:
lr_fs_results = comparar_feature_sets(LinearRegression(), "线性回归")


### 4.2 详细线性回归（含系数输出和散点图）

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score, max_error
from scipy.stats import pearsonr

# StandardScaler：z = (x - μ) / σ（均值0，标准差1）
scaler = StandardScaler()
X_learning_rl = X_learning.copy()
X_test_rl     = X_test.copy()

# 只缩放数值列（类别列已经One-Hot了，值是0/1）
X_learning_rl[num_cols] = scaler.fit_transform(X_learning_rl[num_cols])
X_test_rl[num_cols]     = scaler.transform(X_test_rl[num_cols])

reg = LinearRegression()
cv_strategy = KFold(n_splits=5, shuffle=True, random_state=7)
cv_scores = cross_val_score(reg, X_learning_rl, y_learning,
                             cv=cv_strategy, scoring="neg_root_mean_squared_error")
print(f"交叉验证平均 RMSE: {-cv_scores.mean():.3f}\n")

reg.fit(X_learning_rl, y_learning)
print("训练集 R²:", reg.score(X_learning_rl, y_learning))
print("截距（α）:", reg.intercept_)
print("各特征系数（β）：")
for i, beta in enumerate(reg.coef_):
    print(f"  β{i+1} -> {beta:9.3f}")

def printRegStatistics(truth, preds):
    print("RMSE（越小越好）:", root_mean_squared_error(truth, preds))
    print("MAE（越小越好）:", mean_absolute_error(truth, preds))
    print("R²（越接近1越好）:", r2_score(truth, preds))
    corr, pval = pearsonr(truth, preds)
    print(f"Pearson相关系数: {corr:.4f}（p值={pval:.2e}）")
    print("最大误差:", max_error(truth, preds))

preds = reg.predict(X_test_rl)
printRegStatistics(y_test, preds)

# 散点图：预测值 vs 真实值（理想情况下点应在红色对角线上）
plt.figure(figsize=(7, 7))
plt.scatter(preds, y_test, alpha=0.4)
plt.grid()
plt.plot([0, 100], [0, 100], c="r", linestyle="--", linewidth=2, label="完美预测线")
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.xlabel("预测值")
plt.ylabel("真实值")
plt.title("线性回归：预测值 vs 真实值")
plt.legend()
plt.show()


## 5. KNN 回归（KNeighborsRegressor）

**原理**：找测试样本在训练集中最近的 K 个邻居，预测值 = K 个邻居的 y 均值（或加权均值）

**为什么最需要特征选择（维度诅咒）**：
- 特征越多，所有点之间的欧氏距离趋向相等
- 距离失去意义 → KNN 完全失效
- PCA 降维对 KNN 特别有效

**K 的影响**：K 小 → 过拟合；K 大 → 欠拟合

### 5.1 各特征集对比

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn_fs_results = comparar_feature_sets(KNeighborsRegressor(), "KNN 回归")


### 5.2 GridSearchCV — 在最佳特征集上调优

**参数含义**：
- `n_neighbors`：K 值（邻居数）
- `weights`：uniform = 等权投票；distance = 距离越近权重越大
- `metric`：euclidean = 欧氏距离；manhattan = 曼哈顿距离

In [ ]:
from sklearn.model_selection import GridSearchCV, KFold

# 自动选取该模型对比表中 RMSE 最小的特征集
melhor_knn_fs = knn_fs_results.iloc[0]["特征集"]
print(f"KNN 最佳特征集: {melhor_knn_fs}")

Xl_knn, Xt_knn = FEATURE_SETS[melhor_knn_fs]
sc_knn = StandardScaler()
Xl_knn_s = sc_knn.fit_transform(Xl_knn)
Xt_knn_s = sc_knn.transform(Xt_knn)

param_grid_knn = {
    "n_neighbors": [3, 5, 7, 9, 11, 15],
    "weights":     ["uniform", "distance"],
    "metric":      ["euclidean", "manhattan"],
}
grid_knn = GridSearchCV(
    KNeighborsRegressor(), param_grid_knn,
    scoring="neg_root_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=7),
    n_jobs=-1
)
grid_knn.fit(Xl_knn_s, y_learning)

print("最优参数:", grid_knn.best_params_)
print("交叉验证最优 RMSE:", round(-grid_knn.best_score_, 3))

preds_knn = grid_knn.best_estimator_.predict(Xt_knn_s)
print("\n--- 测试集指标 ---")
print(f"RMSE: {round(np.sqrt(mean_squared_error(y_test, preds_knn)), 3)}")
print(f"MAE:  {round(mean_absolute_error(y_test, preds_knn), 3)}")
print(f"R²:   {round(r2_score(y_test, preds_knn), 3)}")


## 6. 随机森林回归（Random Forest Regressor）

**原理**：
- 训练 N 棵决策树（`n_estimators`）
- 每棵树：① Bootstrap 采样（有放回抽样）的训练子集 ② 每次分裂只考虑随机特征子集
- 预测 = N 棵树的平均

**为什么比单棵DT好**：多棵树平均降低了方差（Bagging原理）

**为什么不太需要特征选择**：随机特征采样本身就是内置的特征筛选机制

### 6.1 各特征集对比

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_fs_results = comparar_feature_sets(
    RandomForestRegressor(random_state=7, n_jobs=-1), "随机森林回归"
)


### 6.2 GridSearchCV — 全特征集调优

**参数含义**：
- `n_estimators`：树的数量（越多越稳定，但训练越慢）
- `max_depth`：每棵树最大深度
- `min_samples_split`：分裂节点所需最少样本数

In [ ]:
param_grid_rf = {
    "n_estimators":      [100, 200],
    "max_depth":         [5, 10, None],
    "min_samples_split": [2, 5, 10],
}
# RF不需要scaling，但统一缩放不影响结果
sc_rf = StandardScaler()
Xl_rf_s = sc_rf.fit_transform(X_learning)
Xt_rf_s = sc_rf.transform(X_test)

grid_rf = GridSearchCV(
    RandomForestRegressor(random_state=7, n_jobs=-1),
    param_grid_rf,
    scoring="neg_root_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=7),
    n_jobs=-1
)
grid_rf.fit(Xl_rf_s, y_learning)

print("最优参数:", grid_rf.best_params_)
print("交叉验证最优 RMSE:", round(-grid_rf.best_score_, 3))

preds_rf = grid_rf.best_estimator_.predict(Xt_rf_s)
print("\n--- 测试集指标 ---")
print(f"RMSE: {round(np.sqrt(mean_squared_error(y_test, preds_rf)), 3)}")
print(f"MAE:  {round(mean_absolute_error(y_test, preds_rf), 3)}")
print(f"R²:   {round(r2_score(y_test, preds_rf), 3)}")


## 7. MLP 回归（多层感知机）

**原理**：输入层 → 隐藏层（非线性激活）→ 输出层（1个神经元，直接输出数值）
使用反向传播 + 梯度下降更新权重。

**激活函数**：
- `relu`：max(0, x)，计算快，不会梯度消失
- `tanh`：输出[-1,1]，对称，适合有负相关的特征

**alpha**：L2正则化系数，越大模型越简单，防止过拟合

### 7.1 各特征集对比

In [ ]:
from sklearn.neural_network import MLPRegressor
import warnings
warnings.filterwarnings("ignore")  # 忽略收敛警告

mlp_fs_results = comparar_feature_sets(
    MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=7),
    "MLP 回归"
)


### 7.2 GridSearchCV — 在最佳特征集上调优网络结构

**参数含义**：
- `hidden_layer_sizes`：网络结构，如(64,32) = 两层，分别64和32个神经元
- `activation`：激活函数（relu / tanh）
- `alpha`：L2正则化强度

In [ ]:
melhor_mlp_fs = mlp_fs_results.iloc[0]["特征集"]
print(f"MLP 最佳特征集: {melhor_mlp_fs}")

Xl_mlp, Xt_mlp = FEATURE_SETS[melhor_mlp_fs]
sc_mlp = StandardScaler()
Xl_mlp_s = sc_mlp.fit_transform(Xl_mlp)
Xt_mlp_s = sc_mlp.transform(Xt_mlp)

param_grid_mlp = {
    "hidden_layer_sizes": [(64,), (64, 32), (128, 64)],
    "activation":         ["relu", "tanh"],
    "alpha":              [0.0001, 0.001],
}
grid_mlp = GridSearchCV(
    MLPRegressor(max_iter=500, random_state=7),
    param_grid_mlp,
    scoring="neg_root_mean_squared_error",
    cv=KFold(n_splits=5, shuffle=True, random_state=7),
    n_jobs=-1
)
grid_mlp.fit(Xl_mlp_s, y_learning)

print("最优参数:", grid_mlp.best_params_)
print("交叉验证最优 RMSE:", round(-grid_mlp.best_score_, 3))

preds_mlp = grid_mlp.best_estimator_.predict(Xt_mlp_s)
print("\n--- 测试集指标 ---")
print(f"RMSE: {round(np.sqrt(mean_squared_error(y_test, preds_mlp)), 3)}")
print(f"MAE:  {round(mean_absolute_error(y_test, preds_mlp), 3)}")
print(f"R²:   {round(r2_score(y_test, preds_mlp), 3)}")


## 8. 最终模型对比

汇总每个模型的最佳结果（最佳特征集 + GridSearchCV最优参数）。

**指标说明**：RMSE ↓ / MAE ↓ / R² ↑（越好的方向）

In [ ]:
resumo = []
for nome_modelo, df_fs in [
    ("决策树回归",   dt_fs_results),
    ("线性回归",     lr_fs_results),
    ("KNN 回归",    knn_fs_results),
    ("随机森林回归", rf_fs_results),
    ("MLP 回归",    mlp_fs_results),
]:
    melhor = df_fs.iloc[0]
    resumo.append({
        "模型":       nome_modelo,
        "最佳特征集": melhor["特征集"],
        "特征数量":   melhor["特征数量"],
        "RMSE":      melhor["RMSE（↓）"],
        "MAE":       melhor["MAE（↓）"],
        "R²":        melhor["R²（↑）"],
    })

df_resumo = pd.DataFrame(resumo).sort_values("RMSE").reset_index(drop=True)
print("=== 所有模型最终对比（回归） ===")
print(df_resumo.to_string(index=False))


## 9. 错误分析（Análise de Erro）

**必做项目**：检查模型预测失败最严重的样本，寻找潜在原因。

使用 **随机森林**（最佳模型）作为参考。

**残差定义**：`残差 = 预测值 - 真实值`
- 残差 > 0 → 预测过高（sobreavaliação，过度估计）
- 残差 < 0 → 预测过低（subestimação，低估）

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 用RF最优模型预测测试集
preds_err    = grid_rf.best_estimator_.predict(Xt_rf_s)
residuos_err = preds_err - y_test.values  # 残差 = 预测 - 真实

print("=== 残差统计 ===")
print(f"  均值（bias）: {residuos_err.mean():.3f}  ← 接近0说明无系统性偏差")
print(f"  标准差:       {residuos_err.std():.3f}")
print(f"  最小值:       {residuos_err.min():.3f}  ← 最大低估")
print(f"  最大值:       {residuos_err.max():.3f}  ← 最大高估")
print(f"  RMSE:         {np.sqrt((residuos_err**2).mean()):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 左图：残差分布（接近正态分布且以0为中心 = 模型无系统偏差）
axes[0].hist(residuos_err, bins=40, edgecolor="black", color="steelblue")
axes[0].axvline(0, color="red", linestyle="--", linewidth=1.5, label="零误差线")
axes[0].set_xlabel("残差（预测 - 真实）")
axes[0].set_ylabel("频次")
axes[0].set_title("残差分布")
axes[0].legend()

# 右图：预测值 vs 真实值（理想：点落在红色对角线上）
axes[1].scatter(preds_err, y_test.values, alpha=0.3, s=10, color="steelblue")
axes[1].plot([0, 100], [0, 100], "r--", linewidth=1.5, label="完美预测线")
axes[1].set_xlabel("预测值")
axes[1].set_ylabel("真实值")
axes[1].set_title("预测值 vs 真实值（随机森林）")
axes[1].legend()

plt.tight_layout()
plt.show()


### 9.1 误差最大的 Top-10 样本

In [ ]:
# 构建包含原始特征的结果DataFrame
df_erros = X_test.copy().reset_index(drop=True)
df_erros["真实值"]   = y_test.values
df_erros["预测值"]   = preds_err
df_erros["残差"]     = residuos_err
df_erros["绝对误差"] = np.abs(residuos_err)

# 找出绝对误差最大的10个样本
top_erros = df_erros.nlargest(10, "绝对误差")[
    ["真实值", "预测值", "残差", "绝对误差",
     "study_hours", "sleep_hours", "mental_health_score",
     "gaming_hours", "social_media_hours", "screen_time_hours"]
].reset_index(drop=True)

print("=== 预测误差最大的 Top-10 样本 ===")
print(top_erros.to_string(index=False))


### 9.2 误差大样本 vs 普通样本 — 特征均值对比

In [ ]:
# 定义"误差大"：绝对误差 > 第90百分位数（前10%最差）
threshold_outlier = np.percentile(df_erros["绝对误差"], 90)
mask_outlier = df_erros["绝对误差"] >= threshold_outlier

feat_analise = ["study_hours", "sleep_hours", "mental_health_score",
                "gaming_hours", "social_media_hours", "caffeine_intake_mg",
                "screen_time_hours", "exercise_minutes"]

df_compare = pd.DataFrame({
    "误差大（前10%）": df_erros[mask_outlier][feat_analise].mean(),
    "普通（后90%）":   df_erros[~mask_outlier][feat_analise].mean(),
}).round(2)
df_compare["差值"] = (df_compare["误差大（前10%）"] - df_compare["普通（后90%）"]).round(2)

print("=== 均值对比：误差大的样本 vs 普通样本 ===")
print(df_compare.to_string())

# 对比柱状图
df_compare[["误差大（前10%）", "普通（后90%）"]].plot(kind="bar", figsize=(12, 4), rot=30)
plt.title("特征均值对比：高误差组 vs 普通组")
plt.ylabel("特征平均值")
plt.tight_layout()
plt.show()


### 9.3 结论

- **高估（残差>0）**：学习时间高但心理健康分低的学生 — 模型因为高`study_hours`预测高分，但实际心理状态拖累了成绩

- **低估（残差<0）**：游戏/社交媒体时间多但成绩仍然好的学生 — 模型对这类习惯的惩罚过重

- 残差分布以0为中心（无系统性偏差），但长尾说明有部分行为组合超出了模型学到的规律

**结论**：最大误差集中在"非典型"学生身上 — 即行为习惯组合罕见的学生（高学习+差心理健康，或高娱乐+高分）

## 10. 可解释性分析 — SHAP

**什么是SHAP**：基于博弈论（Shapley值），量化每个特征对每次预测的贡献

- 正值：该特征使预测值**升高**
- 负值：该特征使预测值**降低**
- 绝对值大：该特征对这次预测影响大

**对比**：BlackBox模型（随机森林）vs 可解释模型（决策树）

**4个例子的选法**（类比分类的TP/TN/FP/FN）：

| 类型 | 选法 |
|---|---|
| 预测准+高分（TP类比）| 残差小 + 真实分数高 |
| 预测准+低分（TN类比）| 残差小 + 真实分数低 |
| 高估（FP类比）| 预测 >> 真实（残差大正值）|
| 低估（FN类比）| 预测 << 真实（残差大负值）|

### 10.1 全局分析 — 随机森林（BlackBox）

In [ ]:
!pip install shap -q


In [ ]:
import shap
import matplotlib.pyplot as plt

print("=== SHAP 可解释性分析（回归）===")

# 使用第6节训练好的随机森林
rf_model_shap = grid_rf.best_estimator_

# 转为DataFrame，保留特征名（SHAP图表需要）
X_test_shap_df  = pd.DataFrame(Xt_rf_s, columns=X_learning.columns)
X_learn_shap_df = pd.DataFrame(Xl_rf_s, columns=X_learning.columns)

# TreeExplainer：专门针对树模型，速度快
explainer_rf   = shap.TreeExplainer(rf_model_shap)
shap_values_rf = explainer_rf.shap_values(X_test_shap_df)

# 柱状图：每个特征的平均|SHAP值|（全局重要性）
shap.summary_plot(shap_values_rf, X_test_shap_df, plot_type="bar", show=False)
plt.title("SHAP — 特征全局重要性（随机森林）")
plt.tight_layout()
plt.show()

# Beeswarm图：每个特征值对预测的影响分布
# 红色=特征值高，蓝色=特征值低；右边=正向影响，左边=负向影响
shap.summary_plot(shap_values_rf, X_test_shap_df, show=False)
plt.title("SHAP Beeswarm — 随机森林")
plt.tight_layout()
plt.show()


### 10.2 选取4个代表性样本

In [ ]:
# 残差 = 预测 - 真实
preds_rf_shap = rf_model_shap.predict(X_test_shap_df)
residuos      = preds_rf_shap - y_test.values
abs_res       = np.abs(residuos)
mediana_y     = np.median(y_test.values)

# 用百分位数定义"准"和"差"
th_bom = np.percentile(abs_res, 25)  # 残差<25th百分位 = 预测准
th_mau = np.percentile(abs_res, 75)  # 残差>75th百分位 = 预测差

# 各取1个样本
idx_tp = int(np.where((abs_res < th_bom) & (y_test.values > mediana_y))[0][0])
idx_tn = int(np.where((abs_res < th_bom) & (y_test.values <= mediana_y))[0][0])
idx_fp = int(np.where(residuos >  th_mau)[0][0])
idx_fn = int(np.where(residuos < -th_mau)[0][0])

exemplos = {
    "预测准+高分（TP类比）": idx_tp,
    "预测准+低分（TN类比）": idx_tn,
    "高估（FP类比）":        idx_fp,
    "低估（FN类比）":        idx_fn,
}

print("=== 4个代表性样本 ===")
for tipo, idx in exemplos.items():
    print(f"  {tipo}: 序号={idx:4d} | 真实={y_test.values[idx]:.1f}"
          f" | 预测={preds_rf_shap[idx]:.1f} | 残差={residuos[idx]:+.1f}")


### 10.3 局部解释 — 随机森林 vs 决策树

**Waterfall图解读**：
- 从基准值（base value = 所有样本的平均预测值）出发
- 每个特征条形表示该特征对这次预测的贡献（红=推高，蓝=拉低）
- 最终值 = 基准值 + 所有特征贡献之和

决策树重新用与RF相同的缩放数据训练（保证特征一致，便于比较）。

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# 用RF相同的缩放数据重新训练DT，保证特征对齐
dt_shap_model = DecisionTreeRegressor(**grid.best_params_, random_state=7)
dt_shap_model.fit(X_learn_shap_df, y_learning)

explainer_dt   = shap.TreeExplainer(dt_shap_model)
shap_values_dt = explainer_dt.shap_values(X_test_shap_df)

ev_rf      = float(np.atleast_1d(explainer_rf.expected_value)[0])
ev_dt      = float(np.atleast_1d(explainer_dt.expected_value)[0])
feat_names = X_learning.columns.tolist()

for tipo, idx in exemplos.items():
    print(f"\n{'='*65}")
    print(f"  {tipo}")
    print(f"  真实={y_test.values[idx]:.1f} | RF预测={preds_rf_shap[idx]:.1f}"
          f" | DT预测={dt_shap_model.predict(X_test_shap_df.iloc[[idx]])[0]:.1f}")

    exp_rf = shap.Explanation(values=shap_values_rf[idx], base_values=ev_rf,
                               data=X_test_shap_df.iloc[idx].values, feature_names=feat_names)
    exp_dt = shap.Explanation(values=shap_values_dt[idx], base_values=ev_dt,
                               data=X_test_shap_df.iloc[idx].values, feature_names=feat_names)

    fig, axes = plt.subplots(1, 2, figsize=(20, 5))
    plt.sca(axes[0])
    shap.plots.waterfall(exp_rf, max_display=10, show=False)
    axes[0].set_title(f"随机森林 — {tipo}", fontsize=10)

    plt.sca(axes[1])
    shap.plots.waterfall(exp_dt, max_display=10, show=False)
    axes[1].set_title(f"决策树 — {tipo}", fontsize=10)

    plt.suptitle(
        f"SHAP Waterfall | 真实={y_test.values[idx]:.1f} "
        f"RF={preds_rf_shap[idx]:.1f} DT={dt_shap_model.predict(X_test_shap_df.iloc[[idx]])[0]:.1f}",
        fontsize=11
    )
    plt.tight_layout()
    plt.show()


### 10.4 分析与假设

**一致性（RF 和 DT 通常同意的）**：
`study_hours` 和 `mental_health_score` 被两个模型识别为最重要的正向特征，因为它们在互信息中排名最高。这种一致性表明该关系是稳健的，不是模型的偶然产物。

**差异性**：
- **高估（FP类比）** 样本：RF 倾向于给`sleep_hours`低 + `gaming_hours`高的非线性组合更高权重，而 DT 因为只走一条分裂路径，可能无法捕捉这种组合效应，预测值更接近真实。

- **低估（FN类比）** 样本：DT 受 `max_depth` 限制，无法学习深层规律；RF 通过多棵树的平均可以捕捉这些 DT 无法覆盖的细节。

**总体假设**：
RF 把预测权重分散给更多特征（ensemble效果），DT 集中在少数主导特征。这种差异在极端样本（高估/低估）中最为明显。